# LoRA Walkthrough

Goal: understand LoRA through rank, parameter count, no-op initialization, and merge-for-inference.


## Paper-to-code map

Official source:

- `learning/lora-family/official/repos/LoRA/loralib/layers.py`
- `learning/lora-family/official/repos/LoRA/loralib/utils.py`

Runnable local source:

- `learning/lora-family/src/lora_minimal.py`


In [ ]:
from __future__ import annotations
import importlib.util
import math
import platform
from pathlib import Path
import torch
REPO = Path.cwd()
if not (REPO / "learning").exists():
    REPO = Path.cwd().parent
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("repo", REPO)
import torch.nn as nn


## Step 1: tiny LoRA linear layer

LoRA adds `delta W = alpha / r * B @ A` to a frozen base weight.


In [ ]:
class TinyLoRALinear(nn.Module):
    def __init__(self, d_in: int, d_out: int, r: int, alpha: int):
        super().__init__()
        self.base = nn.Linear(d_in, d_out, bias=False)
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.empty(r, d_in))
        self.B = nn.Parameter(torch.zeros(d_out, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scaling = alpha / r
    def forward(self, x):
        return self.base(x) + self.scaling * (x @ self.A.T @ self.B.T)
    def merged_weight(self):
        return self.base.weight + self.scaling * (self.B @ self.A)
layer = TinyLoRALinear(6, 4, r=2, alpha=8)
print("W0", tuple(layer.base.weight.shape), "A", tuple(layer.A.shape), "B", tuple(layer.B.shape))
print("trainable", sum(p.numel() for p in layer.parameters() if p.requires_grad))


## Step 2: no-op initialization and merge


In [ ]:
torch.manual_seed(3)
x = torch.randn(5, 6)
assert torch.allclose(layer.base(x), layer(x))
with torch.no_grad():
    layer.B.normal_(0.0, 0.02)
assert torch.allclose(layer(x), x @ layer.merged_weight().T, atol=1e-6)
print("merge equivalence passed")


## Step 3: rank knob


In [ ]:
def lora_param_count(d_in, d_out, r):
    return r * d_in + d_out * r
for r in [1, 2, 4, 8, 16]:
    print({"rank": r, "params": lora_param_count(6, 4, r), "full_delta": 6 * 4})


## Step 4: official source smoke check


In [ ]:
official_root = REPO / "learning" / "lora-family" / "official" / "repos" / "LoRA"
files = [official_root / "loralib" / "layers.py", official_root / "loralib" / "utils.py", official_root / "setup.py"]
for file in files:
    print(file.relative_to(REPO), "exists=", file.exists())
assert all(file.exists() for file in files)


## Closed-book checks

1. Compute LoRA parameter count for `d_in=768`, `d_out=2304`, `r=8`.
2. Explain why zero-initialized `B` makes the wrapper a no-op at initialization.
3. Explain why merge is useful for inference.
